# Cellpose finetuning
A notebook to evaluate finetuning performance of various cellpose fine-tuned models

In [ ]:
import os
import json
from pathlib import Path
import re
from sphero_vem.io import imread_labels_downscaled, imread_downscaled
from dotenv import load_dotenv
import cellpose.metrics as metrics
import numpy as np
import tifffile
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import ListedColormap
import seaborn as sns
from skimage import measure


load_dotenv("../.env")
DATA_ROOT = Path(os.environ.get("DATA_ROOT"))
dir_gt = DATA_ROOT / "processed/labeled/Au_01-vol_01/labeled-01"
dir_pred = DATA_ROOT / "processed/segmented/finetuning"
dir_pretrained = DATA_ROOT / "processed/segmented/Au_01-vol_01/downscaling_tests"
dir_save = DATA_ROOT / "figures/segmentation/finetuning"
dir_save.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "Helvetica"

In [ ]:
# Functions
def calculate_ap(row: pd.Series) -> np.ndarray:
    ground_truth = imread_labels_downscaled(
        row["ground_truth_path"], row["downscaling_factor"]
    )
    predictions = tifffile.imread(row["prediction_path"])
    metric = metrics.average_precision(
        ground_truth, predictions, threshold=row["iou_threshold"]
    )[0]
    return metric


def plot_diff(
    ax,
    ground_truth,
    predictions,
    cmap="coolwarm",
    colorbar_width="5%",
    colorbar_pad=0.1,
):
    """Plot difference between between ground truth and predictions"""

    # Map false positive to -1 and false negatives to +1
    diff_mask = (ground_truth > 0) * 1 - (predictions > 0) * 1

    # Create discrete colormap
    colormap = mpl.colormaps[cmap]
    colors = [colormap(0.05), colormap(0.5), colormap(0.95)]
    discrete_cmap = ListedColormap(colors)

    im = ax.imshow(diff_mask, cmap=discrete_cmap, vmin=-1.5, vmax=1.5)

    # Create colorbar axis with controlled size
    cax = inset_axes(
        ax,
        width=colorbar_width,
        height="100%",
        loc="center left",
        bbox_to_anchor=(1 + colorbar_pad, 0.0, 1, 1),
        bbox_transform=ax.transAxes,
        borderpad=0,
    )

    boundaries = [-1.5, -0.5, 0.5, 1.5]
    cbar = plt.colorbar(
        im, cax=cax, boundaries=boundaries, ticks=[-1, 0, 1], extend="neither"
    )
    cbar.set_ticklabels(["FP", "TP", "FN"])
    for spine in cbar.ax.spines.values():
        spine.set_visible(False)

    return cax


def find_contours(labels: np.ndarray) -> list[np.ndarray]:
    """Find external contours for each predicted label"""
    contours = []
    label_vals = np.unique(labels)[1:]
    for val in label_vals:
        contours_val = measure.find_contours(labels == val)
        contours.append(sorted(contours_val, reverse=True, key=len)[0])
    return contours


def match_predictions(ground_truth: np.ndarray, predictions: np.ndarray) -> np.ndarray:
    """Return prediction masks with matched label indices as ground truth"""
    _, matched = metrics.mask_ious(ground_truth, predictions)
    full_range = np.unique(predictions)[1:]
    missing = np.setdiff1d(full_range, matched).tolist()
    predictions_matched = predictions.copy()
    for val in missing:
        predictions_matched[predictions_matched == val] = 2 * predictions.max() + val

    for i, val in enumerate(matched):
        predictions_matched[predictions_matched == val] = predictions.max() + i + 1

    predictions_matched[predictions_matched > 0] -= predictions.max()

    return predictions_matched

In [ ]:
# Tuple of (model_name, downscaling, ground truth, masks)
targets = []

# Log also ground truths to track the unique ones used before building up df
gt_paths = []

for dir_model in dir_pred.glob("*/"):
    with open(dir_model / "training_manifest.json", "r") as f:
        manifest = json.load(f)
    try:
        n_epochs = manifest["n_epochs"]
    except KeyError:
        n_epochs = pd.NA
    factor_train = manifest["preprocessing_steps"][0]["factor"]
    for masks_path in dir_model.rglob("*.tif"):
        ds_match = re.search(r"downscaled-(\d+)", masks_path.parent.name)
        factor = ds_match.groups()[0] if ds_match else factor_train
        gt_paths.append(dir_gt / "labels" / masks_path.name)
        targets.append(
            (
                dir_model.name,
                int(factor_train),
                n_epochs,
                int(factor),
                gt_paths[-1],
                masks_path,
            )
        )


# Add predictions using pretrained model at different scales, on the same test images
gt_paths = np.unique(gt_paths).tolist()
for gt_path in gt_paths:
    for masks_path in dir_pretrained.glob(f"*{gt_path.stem.replace('-cells', '')}*tif"):
        factor = re.search(r"-ds(\d+)", masks_path.name).groups()[0]
        targets.append(
            (
                "cellposeSAM-pretrained",
                "pretrained",
                "pretrained",
                int(factor),
                gt_path,
                masks_path,
            )
        )

data = pd.DataFrame(
    targets,
    columns=[
        "model_name",
        "downscaling_training",
        "n_epochs",
        "downscaling_factor",
        "ground_truth_path",
        "prediction_path",
    ],
)

data = data.drop_duplicates(
    ["model_name", "downscaling_training", "downscaling_factor", "ground_truth_path"]
)
data = data.drop_duplicates(
    ["model_name", "downscaling_training", "downscaling_factor", "ground_truth_path"]
)
data["iou_threshold"] = [np.arange(start=0.05, stop=1, step=0.05)] * len(data)
data["average_precision"] = data.apply(calculate_ap, axis=1)

# Manually assign missing values
data.loc[data["model_name"] == "cellposeSAM-cells-ds16-20250527_125402", "n_epochs"] = (
    200
)

In [ ]:
data_exploded = data.explode(["iou_threshold", "average_precision"], ignore_index=True)
data_exploded["iou_threshold"] = pd.to_numeric(data_exploded["iou_threshold"])
data_exploded["average_precision"] = pd.to_numeric(data_exploded["average_precision"])

data_long = data_exploded.groupby(
    [
        "model_name",
        "downscaling_training",
        "downscaling_factor",
        "iou_threshold",
        "n_epochs",
    ],
    as_index=False,
)["average_precision"].aggregate("mean")

data_agg = data_long.groupby(
    ["model_name", "downscaling_training", "downscaling_factor", "n_epochs"],
    as_index=False,
)["average_precision"].aggregate("mean")
data_agg = data_agg.rename(columns={"average_precision": "mean_average_precision"})
data_summary = data_agg.groupby(
    ["downscaling_training", "downscaling_factor"], as_index=False
)["mean_average_precision"].aggregate("max")
data_summary["model_type"] = np.where(
    data_summary["downscaling_training"] == "pretrained", "pretrained", "finetuned"
)

In [ ]:
# Performance at training resolutions
data_plot = data_long.loc[
    pd.to_numeric(data_long["downscaling_training"], errors="coerce")
    == data_long["downscaling_factor"]
].copy()
g = sns.relplot(
    data_plot,
    x="iou_threshold",
    y="average_precision",
    style="n_epochs",
    hue="downscaling_training",
    kind="line",
)
g.ax.set(
    xlim=(0.05, 0.95), ylim=(0, 1), xlabel="IoU threshold", ylabel="Average precision"
)
g.savefig(dir_save / "cellposeSAM-finetuning-native-all.png", dpi=300)

In [ ]:
# Performance at native model resolution
factors = [5, 8, 10, 16, 20]
for factor in factors:
    data_plot = data_long.query(
        f"downscaling_training==[{factor}, 'pretrained'] & downscaling_factor=={factor}"
    )
    hue_order = (
        data_plot["n_epochs"].unique()
        if factor != 10
        else [351, 501, 1001, "pretrained"]
    )
    g = sns.relplot(
        data_plot,
        x="iou_threshold",
        y="average_precision",
        hue="n_epochs",
        style="downscaling_training",
        kind="line",
        hue_order=hue_order,
    )
    g.ax.set(
        xlim=(0.05, 0.95),
        ylim=(0, 1),
        xlabel="IoU threshold",
        ylabel="Average precision",
        title=f"Precision of models trained at downscaling factor {factor}",
    )
    g.savefig(dir_save / f"cellposeSAM-finetuning-AP-ds{factor}-native.png", dpi=300)

In [ ]:
data_plot = data_summary.query(
    "(downscaling_factor==downscaling_training | model_type=='pretrained') & downscaling_factor==[5, 8, 10, 16, 20]"
)
g = sns.catplot(
    data_plot,
    x="downscaling_factor",
    y="mean_average_precision",
    kind="bar",
    errorbar=None,
    hue="model_type",
    hue_order=["pretrained", "finetuned"],
)
g.ax.set(
    ylim=(0, 1),
    xlabel="Downscaling factor",
    ylabel="Mean average precision across all thresholds",
    title="Best models at native training resolution",
)
g.refline(y=0.85)
g.ax.yaxis.set_minor_locator(plt.MultipleLocator(0.1))
g.ax.tick_params(axis="y", which="minor", left=True)

g.savefig(dir_save / "cellposeSAM-finetuning-meanAP-native.png", dpi=300)

In [ ]:
for factor in [1, 5, 10]:
    data_plot = data_long.query(f"downscaling_factor=={factor}").copy()
    data_plot["model_type"] = np.where(
        data_plot["downscaling_training"] == "pretrained", "pretrained", "finetuned"
    )
    best_models = data_plot.loc[
        data_plot.groupby("downscaling_training")["n_epochs"].idxmax(), "model_name"
    ].to_list()
    data_plot = data_plot.query("model_name==@best_models")

    g = sns.relplot(
        data_plot,
        x="iou_threshold",
        y="average_precision",
        hue="downscaling_training",
        style="model_type",
        kind="line",
        hue_order=[5, 8, 10, 16, 20, "pretrained"],
    )
    g.ax.set(
        xlim=(0.05, 0.95),
        ylim=(0, 1),
        xlabel="IoU threshold",
        ylabel="Average precision",
        title=f"Precision of models evaluated at downscaling factor {factor}",
    )
    g.savefig(dir_save / f"cellposeSAM-finetuning-AP-upscaled-ds{factor}.png", dpi=300)

In [ ]:
images_data = data.loc[
    (data["model_name"] == "cellposeSAM-cells-ds10-20250605_111901")
    & (data["downscaling_factor"] == 10)
]

row = images_data.iloc[0]
image_path = row["ground_truth_path"].parents[1] / row[
    "ground_truth_path"
].name.replace("-cells", "")
image = imread_downscaled(image_path, row["downscaling_factor"])
ground_truth = imread_labels_downscaled(
    row["ground_truth_path"], row["downscaling_factor"]
)
predictions = tifffile.imread(row["prediction_path"])

In [ ]:
model_name = "cellposeSAM-cells-ds10-20250605_111901"

images_data = data.loc[
    (data["model_name"] == model_name) & (data["downscaling_factor"] == 10)
]

for _, row in images_data.iterrows():
    image_path = row["ground_truth_path"].parents[1] / row[
        "ground_truth_path"
    ].name.replace("-cells", "")
    image = imread_downscaled(image_path, row["downscaling_factor"])
    ground_truth = imread_labels_downscaled(
        row["ground_truth_path"], row["downscaling_factor"]
    )
    predictions = tifffile.imread(row["prediction_path"])

    contours = [find_contours(ground_truth), find_contours(predictions)]
    titles = ["Ground truth", "Predictions", "Difference (ground truth - predictions)"]
    diff_mask = (ground_truth > 0) * 1 - (predictions > 0) * 1

    fig, ax = plt.subplots(1, 3, dpi=250, layout="constrained", figsize=(10, 5))

    for i, contours_image in enumerate(contours):
        ax[i].imshow(image, cmap="gray", vmin=0, vmax=200)
        for contour in contours_image:
            ax[i].plot(contour[:, 1], contour[:, 0], color="y", linewidth=1, ls="--")

    cax = plot_diff(ax[2], ground_truth, predictions, colorbar_pad=0.02)

    for i, axis in enumerate(ax):
        axis.set_axis_off()
        axis.set(title=titles[i])

    fig.savefig(
        dir_save / f"{model_name}-{image_path.stem}-comparison.png",
        bbox_extra_artists=[cax],
        bbox_inches="tight",
        dpi=300,
    )

In [ ]:
data_long.loc[
    (data_long["model_name"] == "cellposeSAM-cells-ds10-20250605_111901")
    & (data_long["downscaling_factor"] == 10)
]